# Stage 2 – Method 1 Margin Loss (Colab)
Contrastive fine-tuning with margin on reconstruction errors.

In [ ]:

import os
import sys
import shutil
from pathlib import Path


use_colab = "google.colab" in sys.modules
if use_colab:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)
    project_dir = Path('/content/drive/MyDrive/liveness_detection_vae')
else:
    project_dir = Path.cwd()


In [ ]:

import os
import sys
import shutil
from pathlib import Path

# project_dir is set in the previous cell
lib_dir = project_dir / 'lib'
sys.path.insert(0, str(lib_dir))
sys.path.insert(0, str(project_dir))

drive_root = project_dir / 'datasets'
drive_zip = drive_root / f"{DATASET_NAME}.zip"
local_root = Path('/content/datasets') if "google.colab" in sys.modules else drive_root
if "google.colab" in sys.modules:
    local_root.mkdir(parents=True, exist_ok=True)
local_zip = local_root / f"{DATASET_NAME}.zip"
extract_dir = local_root / DATASET_NAME
nested_dir = extract_dir / DATASET_NAME

def _has_npz(p: Path) -> bool:
    return p.is_dir() and any(p.rglob('*.npz'))

# Sync zip to local if available
if drive_zip.exists() and (not local_zip.exists() or drive_zip.stat().st_mtime > local_zip.stat().st_mtime):
    shutil.copy2(drive_zip, local_zip)

# Resolve data_dir (prefer local extracted; else extract local zip; else drive extracted)
if _has_npz(extract_dir):
    data_dir = extract_dir
elif _has_npz(nested_dir):
    data_dir = nested_dir
elif local_zip.exists():
    shutil.unpack_archive(str(local_zip), str(local_root))
    if _has_npz(nested_dir):
        data_dir = nested_dir
    elif _has_npz(extract_dir):
        data_dir = extract_dir
    else:
        data_dir = None
else:
    data_dir = None

if data_dir is None:
    drive_extract = drive_root / DATASET_NAME
    drive_nested = drive_extract / DATASET_NAME
    if _has_npz(drive_nested):
        data_dir = drive_nested
    elif _has_npz(drive_extract):
        data_dir = drive_extract

if data_dir is None:
    raise FileNotFoundError(f"Expected {DATASET_NAME} zip/extracted under {drive_root} or local {local_root}")

os.environ['BANDVAE_DATA_DIR'] = str(data_dir)

split_json = project_dir / 'data_split.json'
base_data_dir = data_dir.parent if (data_dir.parent / 'test_balanced_npz').exists() else data_dir
pretrained_path = project_dir / 'runs' / 'stage1_pretrain_colab' / 'stage1_pretrained.pt'
save_dir = project_dir / 'runs' / 'stage2_method1_margin_colab'
save_dir.mkdir(parents=True, exist_ok=True)
os.chdir(project_dir)
print('Split JSON:', split_json)
print('Pretrained:', pretrained_path)
print('Save dir:', save_dir)
print(f'Project dir: {project_dir}')
print(f'Data dir: {data_dir}')


In [ ]:

import time
import torch

import torch.optim as optim
from torch.utils.data import DataLoader

from config_bandvae import get_config
from dataset_stage2 import Stage2Dataset
from model_bandvae import BandSplitVAE, band_split_vae_loss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

config = get_config('full')
config.T_fixed = 300
config.fc_low = 2.0
config.fc_high = 8.0
config.filter_order = 4
config.C_h = 48
config.C_z = 12
config.dilations = [1, 2, 4]
config.lr = 1e-4
config.epochs = 10
config.batch_size = 64
config.num_workers = 4
config.device = device
print(config)


In [ ]:

train_dataset = Stage2Dataset(
    split_json_path=str(split_json),
    split_name='stage2_train',
    T_fixed=config.T_fixed,
    fps=config.fps,
    use_acceleration=config.use_acceleration,
    use_angle=config.use_angle,
    use_angle_rate=config.use_angle_rate,
    fc_low=config.fc_low,
    fc_high=config.fc_high,
    filter_order=config.filter_order,
    random_crop=True,
    base_dir=str(base_data_dir),
)

val_dataset = Stage2Dataset(
    split_json_path=str(split_json),
    split_name='final_test',
    T_fixed=config.T_fixed,
    fps=config.fps,
    use_acceleration=config.use_acceleration,
    use_angle=config.use_angle,
    use_angle_rate=config.use_angle_rate,
    fc_low=config.fc_low,
    fc_high=config.fc_high,
    filter_order=config.filter_order,
    random_crop=False,
    base_dir=str(base_data_dir),
)

train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=config.num_workers,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=config.num_workers,
    pin_memory=True,
)

print('Train samples:', len(train_dataset), 'Val samples:', len(val_dataset))


In [ ]:

checkpoint = torch.load(pretrained_path, map_location=config.device)

model = BandSplitVAE(
    C_in_per_band=config.C_in_per_band,
    C_h=config.C_h,
    C_z=config.C_z,
    dilations=config.dilations,
).to(config.device)
model.load_state_dict(checkpoint['model_state_dict'])

optimizer = optim.Adam(model.parameters(), lr=config.lr)
use_amp = config.device == 'cuda'


In [ ]:


MARGIN = 0.5
LAMBDA_MARGIN = 1.0

def margin_loss(real_loss, fake_loss, margin=MARGIN):
    loss_real = real_loss
    loss_fake = torch.clamp(margin - fake_loss, min=0.0)
    return loss_real + LAMBDA_MARGIN * loss_fake


def validate(model, loader, device, margin):
    model.eval()
    total_loss = real_rec = fake_rec = 0.0
    n_real = n_fake = 0

    with torch.no_grad():
        for x_lf, x_bp, x_hf, labels in loader:
            x_lf = x_lf.to(device)
            x_bp = x_bp.to(device)
            x_hf = x_hf.to(device)
            labels = labels.to(device)

            recons, mus, logvars, x_hat_fused = model(x_lf, x_bp, x_hf)
            targets = {'lf': x_lf, 'bp': x_bp, 'hf': x_hf}
            betas = {'lf': 1.0, 'bp': 1.0, 'hf': 1.0}

            batch_size = x_lf.size(0)
            for i in range(batch_size):
                single_recons = {k: v[i:i+1] for k, v in recons.items()}
                single_mus = {k: v[i:i+1] for k, v in mus.items()}
                single_logvars = {k: v[i:i+1] for k, v in logvars.items()}
                single_targets = {k: v[i:i+1] for k, v in targets.items()}
                single_x_hat = x_hat_fused[i:i+1] if x_hat_fused is not None else None

                loss, loss_dict = band_split_vae_loss(
                    single_recons, single_mus, single_logvars, single_targets, single_x_hat, None,
                    betas=betas, alpha_fusion=0.0
                )

                if labels[i] == 0:
                    real_rec += loss_dict['total']
                    n_real += 1
                else:
                    fake_rec += loss_dict['total']
                    n_fake += 1

    real_mean = real_rec / max(n_real, 1)
    fake_mean = fake_rec / max(n_fake, 1)
    total = margin_loss(real_mean, fake_mean, margin)
    return {
        'total': float(total),
        'real_rec': float(real_mean),
        'fake_rec': float(fake_mean),
        'separation': float(fake_mean - real_mean),
    }


def train_epoch(model, loader, optimizer, device, margin, use_amp=False):
    model.train()
    total_loss = real_rec = fake_rec = 0.0
    n_real = n_fake = 0
    n_batches = len(loader)

    for x_lf, x_bp, x_hf, labels in loader:
        x_lf = x_lf.to(device)
        x_bp = x_bp.to(device)
        x_hf = x_hf.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        real_mask = labels == 0
        fake_mask = labels == 1

        recons, mus, logvars, x_hat_fused = model(x_lf, x_bp, x_hf)
        targets = {'lf': x_lf, 'bp': x_bp, 'hf': x_hf}
        betas = {'lf': 1.0, 'bp': 1.0, 'hf': 1.0}

        if real_mask.any():
            real_loss, real_dict = band_split_vae_loss(
                {k: v[real_mask] for k, v in recons.items()},
                {k: v[real_mask] for k, v in mus.items()},
                {k: v[real_mask] for k, v in logvars.items()},
                {k: v[real_mask] for k, v in targets.items()},
                x_hat_fused[real_mask] if x_hat_fused is not None else None,
                None, betas=betas, alpha_fusion=0.0,
            )
        else:
            real_loss, real_dict = torch.zeros(1, device=device), {'total': 0.0}

        if fake_mask.any():
            fake_loss, fake_dict = band_split_vae_loss(
                {k: v[fake_mask] for k, v in recons.items()},
                {k: v[fake_mask] for k, v in mus.items()},
                {k: v[fake_mask] for k, v in logvars.items()},
                {k: v[fake_mask] for k, v in targets.items()},
                x_hat_fused[fake_mask] if x_hat_fused is not None else None,
                None, betas=betas, alpha_fusion=0.0,
            )
        else:
            fake_loss, fake_dict = torch.zeros(1, device=device), {'total': 0.0}

        loss = margin_loss(real_loss, fake_loss, margin)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        real_rec += float(real_dict['total'])
        fake_rec += float(fake_dict['total'])
        n_real += real_mask.sum().item()
        n_fake += fake_mask.sum().item()

    return {
        'total': total_loss / n_batches,
        'real_rec': real_rec / n_batches,
        'fake_rec': fake_rec / n_batches,
        'separation': (fake_rec - real_rec) / n_batches,
    }

In [ ]:

best_sep = -float('inf')
best_path = save_dir / 'stage2_method1_margin_best.pt'

for epoch in range(1, config.epochs + 1):
    start = time.time()
    train_loss = train_epoch(model, train_loader, optimizer, config.device, MARGIN, use_amp=use_amp)
    val_loss = validate(model, val_loader, config.device, MARGIN)
    duration = (time.time() - start) / 60

    print(
        f"Epoch {epoch}/{config.epochs} | "
        f"train total {train_loss['total']:.4f} real {train_loss['real_rec']:.4f} fake {train_loss['fake_rec']:.4f} sep {train_loss['separation']:.4f} | "
        f"val total {val_loss['total']:.4f} real {val_loss['real_rec']:.4f} fake {val_loss['fake_rec']:.4f} sep {val_loss['separation']:.4f} | "
        f"{duration:.1f} min"
    )

    if val_loss['separation'] > best_sep:
        best_sep = val_loss['separation']
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_metrics': val_loss,
            'config': config,
            'margin': MARGIN,
            'lambda_margin': LAMBDA_MARGIN,
        }, best_path)
        print(f'Saved best to {best_path}')
